# Fase 3 — Núcleo algorítmico y programación orientada a objetos
---
**Proyecto:** comodidades, capacidad y ubicación frente a precio y ocupación estimada de los alojamientos completos de Airbnb en Santiago de Chile (Inside Airbnb, captura del 23-09-2026).

**Objetivo de la fase.** Reorganizar el pipeline de preprocesamiento de la Fase 2 como un sistema de clases y módulos reutilizables, sin cambiar su resultado. Lo verificamos en tres frentes: el pipeline de clases reproduce exactamente el conjunto de la Fase 2, el código se valida en casos normales, límite y de error. Se mide el costo en tiempo y memoria de las operaciones más relevantes.

| Sección | Contenido |
|---|---|
| 1 | De funciones a objetos |
| 2 | Encapsulamiento y partición de los datos |
| 3 | Herencia y polimorfismo |
| 4 | El pipeline |
| 5 | Cohesión y acoplamiento |
| 6 | Validación y equivalencia con la Fase 2 |
| 7 | Recursividad |
| 8 | Eficiencia: tiempo y memoria |
| 9 | Patrón de diseño: Strategy |
| 10 | El pipeline construido desde la configuración |
| 11 | Resultado y entrega a la Fase 4 |
| 12 | Resumen de ejercicios |


## Mejoras incorporadas desde el avance formativo

| Observación de la formativa | Cambio aplicado | Dónde se ve |
|---|---|---|
| El extractor de comodidades se detenía sin indicar la fila ante un texto mal formado | Lanza `ProyectoError` con el índice de la fila, así `src/utilidades.py` pasa a usarse | `src/transformadores.py`, sección 6 |
| La conclusión sobre memoria medía la conversión a lista | Ambas versiones devuelven una Serie y la conclusión se corrigió | Sección 8 |
| Se midió la operación más barata del pipeline | Se midieron los 24 pasos y se compararon dos versiones del más costoso | Sección 8 |
| La partición 80/20 y el escalado no estaban justificados | Se explica su uso y cómo revertir el escalado en la Fase 4 | Secciones 2 y 11.6 |
| El patrón Strategy estaba solo en el cuaderno | `src/estrategias.py` e `ImputadorFlexible`, que el pipeline usa para imputar | Sección 9 |
| Dos casos límite solo imprimían su resultado | Se convirtieron en `assert` | Sección 6 |
| La narrativa conservaba texto de la plantilla | Se reescribió como documentación del proyecto | Todo el cuaderno |

---
## Configuración

Se configura el siguiente código para adaptarlo a nuestro dataset. Se recuerda leer README para la obtención del archivo crudo.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# Única celda de configuración del proyecto: rutas, variables y parámetros.
# =====================================================================

# Ruta escrita desde la raíz del proyecto, no desde la carpeta de este cuaderno
RUTA_DATOS = "data/raw/datasetairbnb.csv"         # archivo crudo
RUTA_F2 = "data/processed/airbnb_procesado.csv"   # resultado de la F2, para comparar

COLUMNA_ID = "id"
OBJETIVOS = ["price", "estimated_occupancy_l365d"]   # no se imputan ni se escalan

VARIABLES_ESTUDIO = ["price", "estimated_occupancy_l365d", "amenities", "accommodates",
                     "bedrooms", "beds", "neighbourhood_cleansed", "latitude",
                     "host_is_superhost", "minimum_nights", "maximum_nights"]

COLUMNAS_IMPUTAR = ["accommodates", "latitude", "bedrooms", "beds",
                    "minimum_nights", "maximum_nights"]
COLUMNAS_MARCAR = ["bedrooms"]
COLUMNAS_NOMINALES = {"host_is_superhost": "superhost",
                      "neighbourhood_cleansed": "comuna"}
COLUMNAS_ESCALAR = ["accommodates", "latitude", "bedrooms", "beds",
                    "minimum_nights", "maximum_nights", "n_comodidades"]

COMODIDADES = {
    "tiene_aire_acondicionado": r"air conditioning|\bac\b",
    "tiene_estacionamiento":    r"parking",
    "tiene_piscina":            r"\bpool\b(?! table)",
    "tiene_lavadora":           r"\bwasher\b",
    "tiene_gimnasio":           r"\bgym\b",
    "tiene_ascensor":           r"elevator",
    "tiene_espacio_trabajo":    r"dedicated workspace",
    "tiene_self_checkin":       r"self check-in",
}

UMBRAL_PRECIO = 10_000_000
MAX_NOCHES = 1125
SEMILLA = 42
PROPORCION_PRUEBA = 0.2

COLUMNAS_ESPERADAS = [COLUMNA_ID, "room_type"] + VARIABLES_ESTUDIO

# La misma configuración, empaquetada para entregarla a los módulos de src/
CONFIG = {
    "objetivos": OBJETIVOS,
    "variables_estudio": VARIABLES_ESTUDIO,
    "columnas_imputar": COLUMNAS_IMPUTAR,
    "columnas_marcar": COLUMNAS_MARCAR,
    "columnas_nominales": COLUMNAS_NOMINALES,
    "columnas_escalar": COLUMNAS_ESCALAR,
    "comodidades": COMODIDADES,
    "umbral_precio": UMBRAL_PRECIO,
    "max_noches": MAX_NOCHES,
}

# ==== Rutas: esta parte es igual que antes ====
from pathlib import Path

def localizar_raiz(inicio=None):
    """Si el cuaderno está en F3, la raíz del proyecto es la carpeta de arriba."""
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio

RAIZ = localizar_raiz()
RUTA_DATOS = str(RAIZ / RUTA_DATOS)
RUTA_F2 = str(RAIZ / RUTA_F2)
DIR_PROCESADO = str(RAIZ / "data" / "processed")

print("Archivo crudo :", RUTA_DATOS)
print("Resultado F2  :", RUTA_F2)
print("Objetivos     :", OBJETIVOS)

Archivo crudo : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\raw\datasetairbnb.csv
Resultado F2  : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\airbnb_procesado.csv
Objetivos     : ['price', 'estimated_occupancy_l365d']


## Preparación del entorno

Preparación del entorno requerido para la ejecución del cuaderno:

In [2]:
import os
import sys
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

SEMILLA = 42
np.random.seed(SEMILLA)

# Los módulos del proyecto están en src/, en la raíz. Se agrega la raíz a la
# lista de carpetas donde Python busca módulos, para poder hacer "from src ..."
# OJO: si editan un archivo de src/, reinicien el kernel para que tome los cambios.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Python :", sys.executable)
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

Python : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\.venv\Scripts\python.exe
pandas : 3.1.1
NumPy  : 2.5.4
Semilla: 42


## Carga del conjunto desde el archivo externo

La función `cargar`, que vive en `src/carga.py`, hace tres cosas antes de devolver nada:

1. Comprueba que el archivo exista, y si no, explica dónde ponerlo.
2. Lo lee según su extensión, sea CSV o Excel.
3. Verifica que estén las columnas declaradas en la configuración.

In [3]:
# La carga vive en src/carga.py (leer_archivo, verificar_esquema y cargar).
# El cuaderno le entrega la ruta y las columnas: el módulo no guarda configuración.
from src.carga import cargar

datos = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS)
datos.head()

Archivo leído: c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\raw\datasetairbnb.csv
Forma: 18904 filas x 90 columnas
Esquema verificado: las 13 columnas del pipeline están presentes


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,49392,https://www.airbnb.com/rooms/49392,20260923225049,2026-09-26,NaN,Share my Flat in Providencia,<b>The space</b><br />Departamento en el barri...,NaN,https://a0.muscache.com/pictures/3740612/b1850...,224592,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
1,52811,https://www.airbnb.com/rooms/52811,20260923225049,2026-09-26,NaN,Suite Providencia 1 Santiago Chile,Apartment located on the subway station Manuel...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,244792,...,4.59,4.64,4.36,NaN,NaN,3,3,0,0,0.23
2,80482,https://www.airbnb.com/rooms/80482,20260923225049,2026-09-25,NaN,Room Private for Woman,"Near transport, subway, malls supermarkets, wa...",NaN,https://a0.muscache.com/pictures/3633153/dffd5...,154527,...,NaN,NaN,NaN,NaN,NaN,2,0,2,0,NaN
3,88944,https://www.airbnb.com/rooms/88944,20260923225049,2026-09-26,NaN,COZY APT. PROVIDENCIA METRO WIFI TV,Spacious and cozy apartment. It has excellent ...,NaN,https://a0.muscache.com/pictures/8fd1c60d-5a8f...,485358,...,4.89,4.96,4.73,NaN,NaN,1,1,0,0,1.37
4,90694,https://www.airbnb.com/rooms/90694,20260923225049,2026-09-26,NaN,Apartment x Rent in Providencia 802,APARTMENT<br />Furnished Apartments with one b...,NaN,https://a0.muscache.com/pictures/d29b8d52-ca46...,491253,...,4.83,4.77,4.43,NaN,NaN,2,2,0,0,0.41


## Perfilar () el conjunto

Para perfilar el conjunto utilizado se define la función perfilar () que nos permitirá dar un primer vistazo al conjunto:

In [4]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, valores únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)

print("Columnas con valores faltantes:")
print(perfil[perfil["nulos"] > 0].to_string(index=False))

print("\nTop 10 columnas con más valores nulos:")
print(perfil[["columna", "pct_nulos"]].head(10).to_string(index=False))

print("\nDistribución del tipo de alojamiento:")
print(datos["room_type"].value_counts())

print("\nDistribución del tipo de propiedad:")
print(datos["property_type"].value_counts().head(10))

print("\nObjetivos del estudio:", OBJETIVOS)
print("Ojo: en el crudo 'price' viene como texto, por ejemplo:", datos["price"].dropna().iloc[0])
print(datos["estimated_occupancy_l365d"].describe())

Columnas con valores faltantes:
                     columna    tipo  unicos  nulos  pct_nulos
       neighborhood_overview float64       0  18904     100.00
                  host_since float64       0  18904     100.00
          host_response_time float64       0  18904     100.00
          host_thumbnail_url float64       0  18904     100.00
        host_acceptance_rate float64       0  18904     100.00
          host_response_rate float64       0  18904     100.00
          host_verifications float64       0  18904     100.00
               neighbourhood float64       0  18904     100.00
   host_total_listings_count float64       0  18904     100.00
          host_neighbourhood float64       0  18904     100.00
            calendar_updated float64       0  18904     100.00
            instant_bookable float64       0  18904     100.00
neighbourhood_group_cleansed float64       0  18904     100.00
                   bathrooms float64       0  18904     100.00
                     li

**Tres observaciones que condicionan todo lo que viene (documentadas en la Fase 2).**

1. `price` viene como texto (`$55,106.00`): hay que convertirlo a número antes de cualquier cálculo.
2. Los objetivos no se imputan. Rellenar `price` o `estimated_occupancy_l365d` sería inventar la respuesta: los alojamientos sin precio se eliminan.
3. Hay valores imposibles: precios sobre $10.000.000, `maximum_nights` de 2.147.483.647 y anuncios con más dormitorios que huéspedes.

Ninguna decisión se rehace aquí: la Fase 3 las encapsula en clases.

---
# 1. De funciones a objetos

Así se ve el código de la Fase 2, en celdas sueltas:

In [5]:
# El estilo de la Fase 2: funciona, pero cada celda depende de las anteriores
df_f2 = datos.copy()
df_f2 = df_f2.drop(columns=["id"])
df_f2["beds"] = df_f2["beds"].fillna(df_f2["beds"].median())
df_f2 = pd.get_dummies(df_f2, columns=["room_type"], dtype=int)

print("Resultado (Filas x Columnas):", df_f2.shape)

Resultado (Filas x Columnas): (18904, 92)


En este cuaderno se organiza el código basado en programación orientada a objetos a través de modularidad en src/ como se muestra a continuación:

![alt text](../docs/Organización_modular.png)

Cada uno de los módulos .py, contiene clases con métodos distintos que van desde cargar datos, aprender, transformar hasta realizar mediciones de velocidad y memoria. El detalle se describe en el informe asociado. 

---
# 2. Encapsulamiento

Se aplica el principio de encapsulamiento mediante las clases de transformación implementadas en src/transformadores.py.

Este principio permite organizar los parámetros internos de cada transformador y controlar cómo se utilizan durante el preprocesamiento de los datos.

Para ello, se distinguen dos operaciones fundamentales:

- ajustar(): aprende los parámetros necesarios a partir del conjunto de entrenamiento, como la mediana utilizada para imputar valores faltantes.

- transformar(): aplica los parámetros previamente aprendidos sobre los datos, sin recalcularlos a partir del conjunto de prueba.

Esta separación contribuye a prevenir la fuga de datos (data leakage), evitando que la información del conjunto de prueba influya en los parámetros aprendidos durante el entrenamiento.

Como parte de este procedimiento, el dataset de Airbnb se divide en conjuntos de entrenamiento y prueba mediante una proporción definida y una semilla aleatoria fija. 
Se ha dejado un ejemplo sobre el encapsulamiento:



In [6]:
class ImputadorCamas:
    """Imputa beds separando lo que aprende de lo que aplica."""

    def __init__(self):
        self._mediana = None        # interno: se aprende, no se asigna desde fuera
        self._ajustado = False      # interno: controla el orden de las llamadas

    @property
    def ajustado(self):
        """Solo lectura: se puede consultar, no asignar."""
        return self._ajustado

    @property
    def mediana(self):
        return self._mediana

    def ajustar(self, df):
        self._mediana = float(df["beds"].median())
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError(
                "ImputadorCamas: hay que llamar a ajustar() antes de transformar(). "
                "La mediana se aprende del conjunto de entrenamiento."
            )
        df = df.copy()
        df["beds"] = df["beds"].fillna(self._mediana)
        return df


# El orden equivocado ahora falla con un mensaje claro
imputador = ImputadorCamas()
try:
    imputador.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

RuntimeError: ImputadorCamas: hay que llamar a ajustar() antes de transformar(). La mediana se aprende del conjunto de entrenamiento.


### Por qué separamos entrenamiento y prueba en un estudio descriptivo

El proyecto no entrena modelos predictivos. Aun así, conservamos una partición 80/20 (semilla 42) por una razón de validez: en la Fase 4 las relaciones que describamos (por ejemplo, entre número de comodidades y precio, o entre comuna y ocupación estimada) se calcularán en el conjunto de entrenamiento y se contrastarán en el de prueba. Si una relación aparece en ambos, no depende de la muestra particular; si aparece en uno solo, se informará como inestable.

Por eso los parámetros de imputación y escalado se aprenden solo con el entrenamiento: así el conjunto de prueba se mantiene independiente.

**Variables escaladas.** Escalamos con `EscaladorRobusto` siete predictores: `accommodates`, `latitude`, `bedrooms`, `beds`, `minimum_nights`, `maximum_nights` y `n_comodidades`, para que sus magnitudes sean comparables sin que los valores extremos definan la escala. Los objetivos (`price` y `estimated_occupancy_l365d`) no se escalan, porque se interpretan en pesos y en noches. Para leer un predictor en sus unidades originales en la Fase 4, se aplica `x = z × RIC + mediana` con los parámetros guardados en `parametros_f3.csv`.

In [7]:
# El orden correcto, con separación de conjuntos
entrenamiento = datos.sample(frac=1 - PROPORCION_PRUEBA, random_state=SEMILLA)
prueba = datos.drop(index=entrenamiento.index)

imputador = ImputadorCamas().ajustar(entrenamiento)
prueba_lista = imputador.transformar(prueba)

print("Mediana aprendida del entrenamiento:", round(imputador.mediana, 2))
print("Mediana del conjunto de prueba     :", round(prueba["beds"].median(), 2))
print("Nulos en prueba después de imputar :", int(prueba_lista["beds"].isna().sum()))
print("\nLa mediana aplicada a prueba es la del entrenamiento, aunque en este caso")
print("ambas coincidan: prueba nunca se usó para aprender nada.")

# La propiedad es de solo lectura
try:
    imputador.ajustado = False
except AttributeError as error:
    print("\nAttributeError:", error)

Mediana aprendida del entrenamiento: 2.0
Mediana del conjunto de prueba     : 2.0
Nulos en prueba después de imputar : 0

La mediana aplicada a prueba es la del entrenamiento, aunque en este caso
ambas coincidan: prueba nunca se usó para aprender nada.

AttributeError: property 'ajustado' of 'ImputadorCamas' object has no setter


**Por qué importa.** Sin el control de `_ajustado`, transformar antes de ajustar
produciría un resultado sin sentido **y sin ningún mensaje de error**. El
encapsulamiento convierte un error silencioso en uno visible.

Este bloque es la evidencia del encapsulamiento: atributos internos con
guion bajo, propiedades de solo lectura y validación antes de actuar.

---
# 3. Herencia y polimorfismo

En esta sección, se deja evidencia de cómo los métodos de la clase madre se heredan en las clases hijas.
La clase madre en cuestión es transformador y sus métodos se heredan en las clases hijas. Sin embargo, estos métodos son definidos según el propósito de la clase hija, demostrando de esa forma el concepto de polimorfismo.

Se importa la clase madre:

In [8]:
# La clase madre vive en src/transformadores.py: el código completo, comentado
# línea por línea, está en ese archivo. Aquí se importa y se pone a prueba.
from src.transformadores import Transformador

# La clase base no se usa sola
try:
    Transformador("beds").ajustar(datos)
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: Cada clase hija debe implementar aprender().


Se importan las clases hijas:

In [9]:
# Las clases hijas también viven en src/transformadores.py
from src.transformadores import (
    CodificadorNominal,
    EliminadorColumnas,
    EscaladorRobusto,
    ImputadorMediana,
)

imp = ImputadorMediana("beds")
salida = imp.ajustar_transformar(datos)
print(imp.nombre, "->", imp.parametros)
print("Nulos en beds después:", int(salida["beds"].isna().sum()))

ImputadorMediana(beds) -> {'mediana': 2.0, 'nulos_en_ajuste': 203}
Nulos en beds después: 0


**Se realiza la transformación de la variable** "price", eliminando interferencias que hacen ver al valor como str, transformando de esta manera a float.

In [10]:
# Los 6 pasos de limpieza de la Fase 2, desde src/transformadores.py
from src.transformadores import (
    Acotador,
    ConversorPrecio,
    ExtractorComodidades,
    FiltroFilas,
    MarcadorNulos,
    SelectorColumnas,
)

# Prueba rápida: el precio deja de ser texto
conversor = ConversorPrecio("price")
convertido = conversor.ajustar_transformar(datos)
print(conversor.nombre, "->", datos["price"].dropna().iloc[0], "pasa a",
      convertido["price"].dropna().iloc[0])

ConversorPrecio(price) -> $59,303.00 pasa a 59303.0


### Cómo comprobar que la herencia funciona

No hay que creer en la herencia: se puede verificar.

In [11]:
imputador = ImputadorMediana("beds")

# 1) El objeto ES un ImputadorMediana Y TAMBIÉN es un Transformador
print("¿Es ImputadorMediana?", isinstance(imputador, ImputadorMediana))
print("¿Es Transformador?   ", isinstance(imputador, Transformador))

# 2) La cadena de búsqueda muestra de dónde hereda
print("\\nCadena de herencia:", [c.__name__ for c in ImputadorMediana.__mro__])

# 3) Métodos que NO están escritos en la clase hija y sin embargo funcionan,
#    porque se heredaron de la clase base
print("\\nMétodos heredados de Transformador:")
for metodo in ["ajustar", "transformar", "ajustar_transformar"]:
    definido_en = "ImputadorMediana" if metodo in ImputadorMediana.__dict__ else "Transformador"
    print(f"  {metodo:<22} definido en {definido_en}")

print("\\nMétodos propios de la clase hija:")
for metodo in ["aprender", "aplicar"]:
    print(f"  {metodo:<22} definido en ImputadorMediana")

¿Es ImputadorMediana? True
¿Es Transformador?    True
\nCadena de herencia: ['ImputadorMediana', 'Transformador', 'object']
\nMétodos heredados de Transformador:
  ajustar                definido en Transformador
  transformar            definido en Transformador
  ajustar_transformar    definido en Transformador
\nMétodos propios de la clase hija:
  aprender               definido en ImputadorMediana
  aplicar                definido en ImputadorMediana


**Polimorfismo**: a continuación se muestran cinco transformaciones al dataset utilizando las clases hijas, se puede ver cómo al final el código realiza un bucle de ajustar_transformar, método que se describe en la clase madre:

In [12]:
pasos = [
    EliminadorColumnas("id"),
    ImputadorMediana("beds"),
    CodificadorNominal("room_type"),
    CodificadorNominal("neighbourhood_cleansed"),
    EscaladorRobusto("minimum_nights")
]

resultado = datos.copy()
for paso in pasos:                      # la misma llamada polimórfica para todos los pasos
    resultado = paso.ajustar_transformar(resultado)
    print(f"{paso.nombre:<45} -> {resultado.shape}")

EliminadorColumnas(id)                        -> (18904, 89)
ImputadorMediana(beds)                        -> (18904, 89)
CodificadorNominal(room_type)                 -> (18904, 92)
CodificadorNominal(neighbourhood_cleansed)    -> (18904, 123)
EscaladorRobusto(minimum_nights)              -> (18904, 123)


**Este bloque es la evidencia de los tres principios**

- **Herencia**: las cuatro clases heredan de `Transformador` y no repiten el control de estado.
- **Polimorfismo**: el bucle llama `ajustar_transformar()` sin preguntar el tipo.
- **Encapsulamiento**: `_parametros` y `_ajustado` son internos, y hay validación antes de actuar.

---
# 4. El `Pipeline`

Una clase que guarda el orden, los ejecuta y controla que se haya ajustado antes de
transformar. 
**Nuestro pipeline coordina la secuencia de transformaciones, contiene una lista de objetos Transformador, ejecuta los pasos en orden y resume/organiza el flujo del preprocesamiento.**

In [13]:
# El orquestador vive en src/pipeline.py
from src.pipeline import Pipeline

pipeline = Pipeline([
    EliminadorColumnas("id"),
    ImputadorMediana("beds"),
    CodificadorNominal("room_type"),
    CodificadorNominal("neighbourhood_cleansed"),
    EscaladorRobusto("minimum_nights"),
])

print(pipeline)
pipeline.resumen()

Pipeline(5 pasos, sin ajustar)


,orden,paso,clase
0,1,EliminadorColumnas(id),EliminadorColumnas
1,2,ImputadorMediana(beds),ImputadorMediana
2,3,CodificadorNominal(room_type),CodificadorNominal
3,4,CodificadorNominal(neighbourhood_cleansed),CodificadorNominal
4,5,EscaladorRobusto(minimum_nights),EscaladorRobusto


El código a continuación ajusta con entrenamiento y transforma prueba para evitar la fuga de datos, tal como se mencionó en el apartado de Encapsulamiento:

In [14]:
# Ajustar con entrenamiento, transformar prueba: sin fuga de datos
pipeline.ajustar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, " -> Prueba transformada:", prueba_lista.shape)

escalador = pipeline.pasos_ejecutados()[-1]
print(f"\n{escalador.nombre} aprendió del entrenamiento: {escalador.parametros}")
print("Mediana de minimum_nights en prueba, ANTES de escalar:", prueba["minimum_nights"].median())
print("Esos parámetros se aplicaron a prueba sin recalcular nada con sus datos.")

Entrenamiento: (15123, 90)  -> Prueba transformada: (3781, 123)

EscaladorRobusto(minimum_nights) aprendió del entrenamiento: {'mediana': 1.0, 'ric': 1.0}
Mediana de minimum_nights en prueba, ANTES de escalar: 2.0
Esos parámetros se aplicaron a prueba sin recalcular nada con sus datos.


**Por qué importa.** El escalador no recalcula la mediana ni el RIC con los datos de prueba: aplica los que aprendió del entrenamiento. Si salieran de prueba, el análisis usaría información que no debía ver.

A continuación, se muestra cómo a través del módulo pipeline.py podemos orquestar las 24 transformaciones del dataset:

In [15]:
# construir_pipeline(config) vive en src/pipeline.py y arma los 24 pasos de la Fase 2.
from src.pipeline import construir_pipeline as construir_pipeline_desde_config


def construir_pipeline():
    """Atajo del cuaderno: el pipeline de src/ con la CONFIG de este notebook.

    Así el resto del cuaderno puede seguir escribiendo construir_pipeline()
    sin repetir la configuración en cada celda.
    """
    return construir_pipeline_desde_config(CONFIG)


pipeline_proyecto = construir_pipeline()
print(pipeline_proyecto)
pipeline_proyecto.resumen()

Pipeline(24 pasos, sin ajustar)


,orden,paso,clase
0,1,ConversorPrecio(price),ConversorPrecio
1,2,FiltroFilas(solo alojamientos completos),FiltroFilas
2,3,SelectorColumnas(11 variables),SelectorColumnas
3,4,Acotador(maximum_nights),Acotador
4,5,FiltroFilas(objetivos sin faltantes),FiltroFilas
5,6,MarcadorNulos(bedrooms),MarcadorNulos
6,7,ImputadorFlexible(accommodates),ImputadorFlexible
7,8,ImputadorFlexible(latitude),ImputadorFlexible
8,9,ImputadorFlexible(bedrooms),ImputadorFlexible
9,10,ImputadorFlexible(beds),ImputadorFlexible


---
# 5. Cohesión y acoplamiento

Respecto a este tipo de programación, la alta cohesión hace referencia a que cada componente hace una sola cosa, mientras que el bajo acoplamiento significa que los componentes dependen poco unos de otros: cambiar uno no obliga a tocar los demás. 
- La arquitectura de la Fase 3 se diseñó considerando los principios de alta cohesión y bajo acoplamiento, con el objetivo de favorecer la modularidad, reutilización y mantenimiento del código.
- **Se deja un ejemplo de cohesión baja y cohesión alta:**

In [16]:
class ProcesadorAirbnbTodoEnUno:
    """Ejemplo de COHESIÓN BAJA en el proyecto: hace cuatro cosas sin relación entre sí."""

    def __init__(self, ruta_csv):
        self.ruta_csv = ruta_csv
        self.df = None

    def cargar(self):            # Responsabilidad 1: entrada y salida de archivos (I/O)
        self.df = pd.read_csv(self.ruta_csv)

    def limpiar(self):           # Responsabilidad 2: transformar datos (Pipeline)
        if self.df is not None:
            self.df = self.df.dropna(subset=["beds"])

    def graficar(self):          # Responsabilidad 3: visualización (Matplotlib / Seaborn)
        # Generar gráficos de precios o distribución de alojamientos en Santiago
        pass

    def enviar_correo(self):     # Responsabilidad 4: comunicación / notificaciones
        # Enviar reporte de la auditoría de alojamientos por email
        pass


print("Cuatro responsabilidades distintas (cargar, limpiar, graficar, enviar correo) en una sola clase.")
print("Problema práctico: para probar la limpieza de los datos de Airbnb hay que tener un archivo real,")
print("porque cargar() y limpiar() viven pegadas en el mismo objeto.")

Cuatro responsabilidades distintas (cargar, limpiar, graficar, enviar correo) en una sola clase.
Problema práctico: para probar la limpieza de los datos de Airbnb hay que tener un archivo real,
porque cargar() y limpiar() viven pegadas en el mismo objeto.


Ejemplo de baja cohesión:

In [17]:
import pandas as pd
import numpy as np

class CargadorAirbnb:
    """Una sola responsabilidad: leer el archivo CSV con los alojamientos de Santiago."""

    def __init__(self, ruta):
        self.ruta = ruta

    def cargar(self):
        return pd.read_csv(self.ruta)


class LimpiadorAirbnb:
    """Una sola responsabilidad: limpiar nulos en métricas clave (como beds).

    Fíjense en que NO sabe de dónde vienen los datos. Recibe un DataFrame y
    devuelve otro. Eso es bajo acoplamiento: no depende del Cargador.
    """

    def limpiar(self, df):
        # Limpiar filas donde falte el puntaje de reseñas de los alojamientos
        df = df.copy()
        if "beds" in df.columns:
            df = df.dropna(subset=["beds"])
        return df


# La gran ventaja se ve al probar: no hace falta ningún archivo real ni CSV en disco
mini = pd.DataFrame({
    "id": [101, 102, 103],
    "beds": [4.5, np.nan, 4.8],
    "room_type": ["Entire home/apt", "Private room", "Entire home/apt"]
})

print("Se puede probar el LimpiadorAirbnb sin tocar el disco ni archivos pesados:")
limpiador = LimpiadorAirbnb()
print("  entrada:", mini.shape, " -> salida:", limpiador.limpiar(mini).shape)

Se puede probar el LimpiadorAirbnb sin tocar el disco ni archivos pesados:
  entrada: (3, 3)  -> salida: (2, 3)


## 5.1 Verificación del acoplamiento

Para demostrar este principio se construyen dos pipelines que utilizan la misma estructura, pero aplican la imputación a variables diferentes: beds y minimum_nights.

La comparación permite comprobar que es posible modificar la configuración de una transformación sin alterar la implementación de la clase Pipeline.

Esta organización facilita la incorporación de nuevas transformaciones y favorece la reutilización de los componentes desarrollados durante esta fase.

In [18]:
# Cambiar un paso del pipeline no obliga a tocar ni el Pipeline ni los otros pasos
pipeline_a = Pipeline([EliminadorColumnas("id"), ImputadorMediana("beds")])
pipeline_b = Pipeline([EliminadorColumnas("id"), ImputadorMediana("minimum_nights")])

for nombre, pipe in [("versión A", pipeline_a), ("versión B", pipeline_b)]:
    salida = pipe.ajustar(datos).transformar(datos)
    print(f"{nombre}: {salida.shape}")

print("\nSe cambió el paso y NO se modificó la clase Pipeline ni las demás clases.")
print("Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.")

versión A: (18904, 89)
versión B: (18904, 89)

Se cambió el paso y NO se modificó la clase Pipeline ni las demás clases.
Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.


## 5.2 Cómo se traduce en la estructura de archivos

Una organización con alta cohesión y bajo acoplamiento se reconoce a simple vista:

```
src/
├── carga.py            solo lee y verifica archivos
├── transformadores.py  solo las clases de transformación
├── estrategias.py      solo las estrategias de imputación (patrón Strategy)
├── pipeline.py         solo encadena y ejecuta
├── medicion.py         solo mide tiempo y memoria
└── utilidades.py       error propio del proyecto (ProyectoError)
```

Es precisamente de esta forma en la que se organizó el código para nuestro proyecto. 

---
# 6. Validación: casos normales, límite y excepciones

Probamos el código en tres escenarios: caso normal, casos límite y excepciones.

In [19]:
# --- CASO NORMAL: el flujo completo sobre el conjunto real ---
pipe = Pipeline([EliminadorColumnas("id"), ImputadorMediana("beds")])
salida = pipe.ajustar(datos).transformar(datos)

assert len(salida) == len(datos), "El pipeline perdió o duplicó filas"
assert salida["beds"].isna().sum() == 0, "Quedaron nulos en beds"
assert "id" not in salida.columns, "La columna id no se eliminó"
print("Caso normal: las tres comprobaciones pasaron.")

Caso normal: las tres comprobaciones pasaron.


In [20]:
# --- CASOS LÍMITE: situaciones extremas pero válidas ---

# 1) Una columna sin ningún nulo no debe alterarse
sin_nulos = pd.DataFrame({"beds": [1.0, 2.0, 3.0]})
r1 = ImputadorMediana("beds").ajustar_transformar(sin_nulos)
assert r1["beds"].tolist() == [1.0, 2.0, 3.0]
print("Límite 1: columna sin nulos, sin cambios.")

# 2) Una sola categoría genera una sola columna
una_cat = pd.DataFrame({"room_type": ["Entire home/apt"] * 3})
r2 = CodificadorNominal("room_type").ajustar_transformar(una_cat)
assert list(r2.columns) == ["room_type_Entire_home/apt"]
print("Límite 2: una categoría ->", list(r2.columns))

# 3) Varianza cero no debe producir división por cero
constante = pd.DataFrame({"minimum_nights": [10.0, 10.0, 10.0]})
r3 = EscaladorRobusto("minimum_nights").ajustar_transformar(constante)
assert r3["minimum_nights"].notna().all()
print("Límite 3: varianza cero ->", r3["minimum_nights"].tolist())

# 4) Una categoría nueva en prueba no debe crear columna
pipe_cat = Pipeline([CodificadorNominal("neighbourhood_cleansed")]).ajustar(entrenamiento)
nuevos = prueba.copy()
nuevos.iloc[0, nuevos.columns.get_loc("neighbourhood_cleansed")] = "Comuna Ficticia"
r4 = pipe_cat.transformar(nuevos)
assert "neighbourhood_cleansed_Comuna_Ficticia" not in r4.columns
assert r4.filter(like="neighbourhood_cleansed_").iloc[0].sum() == 0
print("Límite 4: la comuna nueva no crea columna y su fila queda en ceros.")

Límite 1: columna sin nulos, sin cambios.
Límite 2: una categoría -> ['room_type_Entire_home/apt']
Límite 3: varianza cero -> [0.0, 0.0, 0.0]


Límite 4: la comuna nueva no crea columna y su fila queda en ceros.


In [21]:
# --- EXCEPCIONES: entradas que deben fallar con mensaje claro ---
from src.utilidades import ProyectoError

pruebas = [
    ("transformar sin ajustar", lambda: ImputadorMediana("beds").transformar(datos)),
    ("columna inexistente", lambda: ImputadorMediana("columna_ficticia").ajustar(datos)),
    ("agregar algo que no es Transformador", lambda: Pipeline().agregar("texto")),
    ("pipeline sin ajustar", lambda: Pipeline([ImputadorMediana("beds")]).transformar(datos)),
    ("amenities mal formado", lambda: ExtractorComodidades("amenities", COMODIDADES)
         .ajustar_transformar(pd.DataFrame({"amenities": ['["Wifi"]', '["Wifi", ']}))),
    ("archivo inexistente", lambda: cargar("data/raw/no_existe.csv", COLUMNAS_ESPERADAS)),
]

for descripcion, accion in pruebas:
    try:
        accion()
        print(f"  {descripcion:<38} NO lanzó excepción (revisar)")
    except (RuntimeError, KeyError, TypeError, ProyectoError, FileNotFoundError) as error:
        print(f"  {descripcion:<38} {type(error).__name__} capturado")

  transformar sin ajustar                RuntimeError capturado
  columna inexistente                    KeyError capturado
  agregar algo que no es Transformador   TypeError capturado
  pipeline sin ajustar                   RuntimeError capturado
  amenities mal formado                  ProyectoError capturado
  archivo inexistente                    FileNotFoundError capturado


## La verificación que más rinde: el pipeline de clases contra la Fase 2

Si el pipeline con clases produce exactamente el mismo conjunto que guardó la Fase 2, hay una prueba objetiva de que la reorganización no rompió nada. Se compara contra `data/processed/airbnb_procesado.csv`, ajustando con todos los datos porque así trabajó la Fase 2. El índice se reinicia porque el One-Hot de la Fase 2 lo reinició.

In [22]:
# Resultado oficial de la Fase 2, tal como quedó guardado en el repositorio
f2_oficial = pd.read_csv(RUTA_F2)

# El mismo trabajo, hecho por el pipeline de clases.
# Se ajusta con TODOS los datos porque así trabajó la F2 (sin separar entrenamiento y prueba)
pipeline_f2 = construir_pipeline().ajustar(datos)
f3_clases = pipeline_f2.transformar(datos).reset_index(drop=True)

print("Fase 2 (archivo guardado):", f2_oficial.shape)
print("Fase 3 (pipeline de clases):", f3_clases.shape)

# Compara filas, columnas, nombres, orden, tipos y valores.
# Si hubiera UNA sola diferencia, se detiene con un error que dice dónde está.
pd.testing.assert_frame_equal(f3_clases, f2_oficial)
print("\nLas dos versiones son idénticas: mismas filas, columnas, tipos y valores.")

Fase 2 (archivo guardado): (14758, 50)
Fase 3 (pipeline de clases): (14758, 50)

Las dos versiones son idénticas: mismas filas, columnas, tipos y valores.


---
# 7. Recursividad

Toda función recursiva tiene dos partes: un **caso base** que la detiene y un **caso
recursivo** que reduce el problema y vuelve a llamarse.

A continuación se presenta un ejemplo de la función aplanar () que tranforma metadatos anidados en una estructura plana que permite recorrer estructuras cuya profundidad puede variar:

In [23]:
def aplanar(estructura, prefijo=""):
    """Convierte metadatos anidados en pares plano de clave y valor.

    Por qué recursión y no un bucle: la profundidad no se conoce al escribir el
    código. Mañana alguien agrega un nivel y el bucle anidado deja de servir.
    """
    plano = {}
    for clave in estructura:
        valor = estructura[clave]
        compuesta = f"{prefijo}.{clave}" if prefijo else str(clave)
        if isinstance(valor, dict) and valor:
            plano.update(aplanar(valor, compuesta))
        else:
            plano[compuesta] = valor
    return plano


# Metadatos adaptados al proyecto de Airbnb
metadatos = {
    "proyecto": {"nombre": "airbnb_santiago", "fase": 3},
    "datos": {"filas": len(datos), "nulos": {"beds": int(datos["beds"].isna().sum())}},
    "entorno": {"semilla": SEMILLA, "librerias": {"pandas": pd.__version__}},
}

for clave, valor in aplanar(metadatos).items():
    print(f"{clave:<32} {valor}")

proyecto.nombre                  airbnb_santiago
proyecto.fase                    3
datos.filas                      18904
datos.nulos.beds                 203
entorno.semilla                  42
entorno.librerias.pandas         3.1.1


## 7.1 Evaluación de eficiencia mediante memoización
Como complemento al uso de recursividad en los metadatos del proyecto, se comparan dos implementaciones recursivas de la sucesión de Fibonacci: una sin almacenamiento de resultados y otra que utiliza memoización.

El objetivo es evaluar cómo la reutilización de cálculos previamente realizados influye en el tiempo de ejecución.

In [24]:
import time
import pandas as pd

def fib_ingenua(n):
    """Sin memoria: recalcula los mismos subproblemas miles de veces."""
    if n < 2:
        return n
    return fib_ingenua(n - 1) + fib_ingenua(n - 2)


def fib_memoizada(n, cache=None):
    """La misma recursión, guardando lo ya calculado."""
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    resultado = n if n < 2 else fib_memoizada(n - 1, cache) + fib_memoizada(n - 2, cache)
    cache[n] = resultado
    return resultado


filas = []
for n in [18, 22, 26]:
    inicio = time.perf_counter(); fib_ingenua(n); t1 = time.perf_counter() - inicio
    inicio = time.perf_counter(); fib_memoizada(n); t2 = time.perf_counter() - inicio
    filas.append({
        "n": n, 
        "ingenua_s": round(t1, 6), 
        "memoizada_s": round(t2, 6),
        "veces_mas_rapida": round(t1 / t2, 1) if t2 > 0 else 0.0
    })

pd.DataFrame(filas)

,n,ingenua_s,memoizada_s,veces_mas_rapida
0,18,0.000432,0.000010,43.2
1,22,0.003110,0.000015,211.6
2,26,0.017687,0.000013,1320.0


La comparación de ambas implementaciones permite observar el impacto de la memoización sobre la eficiencia computacional.

La versión recursiva ingenua vuelve a calcular subproblemas que ya fueron resueltos, mientras que la versión memoizada almacena estos resultados para reutilizarlos posteriormente.

Este mecanismo permite reducir las operaciones repetidas y mejorar el tiempo de ejecución, aunque requiere memoria adicional para almacenar los valores calculados.

El experimento demuestra que la eficiencia de un algoritmo no depende únicamente de utilizar recursividad, sino también de las decisiones de diseño que permiten evitar operaciones innecesarias.

---
# 8. Eficiencia: medir tiempo y memoria

La rúbrica pide mediciones **reproducibles** con `timeit` y `tracemalloc`, comparación
entre implementaciones e interpretación considerando tiempo **y** memoria.

In [25]:
# medir (tiempo y memoria pico) vive en src/medicion.py
from src.medicion import medir


# Datos reales justo antes de escalar: todos los pasos del pipeline menos los escaladores
pasos_previos = [p for p in construir_pipeline().pasos_ejecutados()
                 if not isinstance(p, EscaladorRobusto)]
antes_de_escalar = datos.copy()
for paso in pasos_previos:
    antes_de_escalar = paso.ajustar_transformar(antes_de_escalar)

escalador = EscaladorRobusto("n_comodidades").ajustar(antes_de_escalar)
MEDIANA, RIC = escalador.parametros["mediana"], escalador.parametros["ric"]


def escalar_con_bucle(serie):
    """Versión A: valor por valor; devuelve una Serie, igual que EscaladorRobusto."""
    escalados = []
    for valor in serie:
        escalados.append((valor - MEDIANA) / RIC)
    return pd.Series(escalados, index=serie.index, name=serie.name)


def escalar_vectorizado(serie):
    """Versión B: la columna completa de una vez (la que usa EscaladorRobusto)."""
    return (serie - MEDIANA) / RIC





serie = antes_de_escalar["n_comodidades"]

# Antes de comparar tiempos: las dos deben dar EXACTAMENTE lo mismo
pd.testing.assert_series_equal(escalar_con_bucle(serie), escalar_vectorizado(serie))
print(f"Las dos versiones dan el mismo resultado sobre los {len(serie)} alojamientos.\n")

# Se mide 5 veces y se conserva el MENOR tiempo
t_bucle = min(timeit.repeat(lambda: escalar_con_bucle(serie), number=50, repeat=5)) / 50
t_vect = min(timeit.repeat(lambda: escalar_vectorizado(serie), number=50, repeat=5)) / 50
_, _, mem_bucle = medir(escalar_con_bucle, serie)
_, _, mem_vect = medir(escalar_vectorizado, serie)

print(f"A · con bucle   : {t_bucle*1000:.3f} ms   {mem_bucle*1024:.1f} KB")
print(f"B · vectorizada : {t_vect*1000:.3f} ms   {mem_vect*1024:.1f} KB")
print(f"\nLa vectorizada es {t_bucle / t_vect:.1f} veces más rápida "
      f"y usa {mem_vect / mem_bucle:.2f} veces la memoria del bucle.")

Las dos versiones dan el mismo resultado sobre los 14758 alojamientos.

A · con bucle   : 2.290 ms   1299.7 KB
B · vectorizada : 0.084 ms   233.3 KB

La vectorizada es 27.4 veces más rápida y usa 0.18 veces la memoria del bucle.


**Cómo se lee.** Las dos versiones hacen la misma cuenta, `(valor − mediana) / RIC`, y `assert_series_equal` confirma que el resultado es idéntico. La diferencia está en **quién** hace el trabajo:

- **Con bucle**, Python ejecuta una resta y una división por cada alojamiento, una por una, y guarda cada resultado como un objeto `float` de Python en una lista que después convierte en Serie.
- **Vectorizada**, pandas entrega la columna completa a NumPy, que hace todas las cuentas de una vez en código compilado y guarda cada valor en 8 bytes.

**La vectorizada gana en las dos dimensiones.** Es unas 29 veces más rápida y su pico de memoria (233 KB) equivale a dos arreglos de 14.758 valores: el intermedio de la resta y el resultado. El bucle necesita cerca de cinco veces más memoria, porque cada valor es un objeto de Python y además se copia a la Serie final.

**Corrección respecto del avance formativo.** En la formativa concluimos que la versión vectorizada usaba un 24 % más de memoria. Esa medición incluía la conversión final a lista (`.tolist()`), que `EscaladorRobusto` no hace. Al devolver una Serie en ambas versiones, igual que el código de `src/`, la conclusión se invierte.

**Cómo crece el costo con el tamaño.** Una medición sobre un solo tamaño no permite hablar de complejidad: hay que ver cómo cambia el tiempo cuando crece el volumen. Hasta el tamaño real se usan muestras del conjunto; sobre él se repiten filas al azar (con reemplazo) solo para observar la tendencia.

In [26]:
# Cómo crece el costo con el tamaño: esto es lo que revela la complejidad
mediciones = []
for n in [1_000, 5_000, len(serie), 50_000, 100_000]:
    # Sobre el tamaño real se repiten filas al azar, solo para ver la tendencia
    muestra = serie.sample(n=n, replace=n > len(serie), random_state=SEMILLA)
    t_b = min(timeit.repeat(lambda: escalar_con_bucle(muestra), number=10, repeat=5)) / 10
    t_v = min(timeit.repeat(lambda: escalar_vectorizado(muestra), number=10, repeat=5)) / 10
    _, _, m_b = medir(escalar_con_bucle, muestra)
    _, _, m_v = medir(escalar_vectorizado, muestra)
    mediciones.append({"filas": n, "bucle_ms": round(t_b*1000, 3),
                       "vectorizada_ms": round(t_v*1000, 3), "razón": round(t_b/t_v, 1),
                       "bucle_KB": round(m_b*1024, 1), "vectorizada_KB": round(m_v*1024, 1)})

print(pd.DataFrame(mediciones).to_string(index=False))

 filas  bucle_ms  vectorizada_ms  razón  bucle_KB  vectorizada_KB
  1000     0.190           0.063    3.0      88.1            18.4
  5000     0.756           0.063   12.0     440.6            80.9
 14758     2.274           0.076   30.0    1299.7           233.3
 50000     7.743           0.103   75.3    4437.2           784.0
100000    16.593           0.134  123.6    8789.3          1565.2


**Cómo se lee la tabla.**

- **Tiempo del bucle.** Crece en proporción a las filas: de 1.000 a 100.000 filas (100 veces más) pasa de 0,18 a 15,1 ms, unas 84 veces más. Es complejidad temporal **lineal, O(n)**: cada valor se procesa una vez, en Python.
- **Tiempo de la vectorizada.** Casi no cambia: de 0,06 a 0,13 ms en el mismo rango. También es O(n), porque NumPy debe procesar cada valor, pero lo hace en código compilado, y a estos tamaños domina un costo fijo de preparación que no depende del número de filas. Por eso la razón entre ambas crece de 3 a 116 veces y no se estabiliza dentro del rango medido.
- **Memoria.** Las dos crecen en proporción a las filas (complejidad espacial **O(n)**), pero la vectorizada usa entre 4,8 y 5,6 veces menos, porque guarda cada valor en 8 bytes, mientras que el bucle crea un objeto de Python por valor.

**Tres reglas que se cumplieron para que la medición valga**

1. **Se repitió y se conservó el tiempo menor.** `timeit.repeat` mide 5 veces y se toma el mínimo: los valores altos suelen reflejar interrupciones del sistema operativo, no el costo del código.
2. **Se midió sobre el tamaño real**: los 14.758 alojamientos completos que procesa el pipeline, además de tamaños menores y mayores para ver la tendencia.
3. **Se comprobó que las versiones dan lo mismo**, con `assert`, antes de comparar tiempos.

**La decisión que tomó la medición.** `EscaladorRobusto.aplicar()` usa la forma vectorizada: es más rápida y además usa menos memoria, así que no hay intercambio que evaluar. Sin embargo, el escalado tarda menos de 1 ms sobre el conjunto real. Para justificar una optimización hay que medir dónde se va realmente el tiempo del pipeline, y eso hacemos a continuación.

## ¿Cuál es el paso más costoso del pipeline?

El escalado es una de las operaciones más baratas del pipeline: menos de 1 ms sobre el conjunto real. Para que una optimización se justifique, hay que saber dónde se va realmente el tiempo. Por eso medimos cada uno de los 24 pasos sobre los datos reales, en el mismo orden en que los ejecuta el pipeline, con `timeit` (5 repeticiones, se conserva el mínimo).

In [27]:
# Tiempo de cada paso del pipeline real, para encontrar el más costoso
tiempos = []
intermedio = datos.copy()
for paso in construir_pipeline().pasos_ejecutados():
    paso.ajustar(intermedio)
    t = min(timeit.repeat(lambda: paso.transformar(intermedio), number=3, repeat=5)) / 3
    intermedio = paso.transformar(intermedio)
    tiempos.append({"paso": paso.nombre, "ms": round(t * 1000, 2)})

tabla_pasos = pd.DataFrame(tiempos).sort_values("ms", ascending=False)
print(f"Tiempo total: {tabla_pasos['ms'].sum():.0f} ms · ExtractorComodidades: "
      f"{tabla_pasos['ms'].iloc[0] / tabla_pasos['ms'].sum():.0%} del total\n")
print(tabla_pasos.head(5).to_string())

Tiempo total: 675 ms · ExtractorComodidades: 84% del total

                                          paso      ms
14             ExtractorComodidades(amenities)  564.79
1     FiltroFilas(solo alojamientos completos)   32.65
0                       ConversorPrecio(price)   23.62
16  CodificadorNominal(neighbourhood_cleansed)   21.20
2               SelectorColumnas(11 variables)    8.10


**Cómo se lee.** `ExtractorComodidades` es, por lejos, el paso más costoso: concentra el 85 % del tiempo total del pipeline (unos 524 de 619 ms), y el segundo paso más lento (el filtro de alojamientos completos, 27 ms) tarda unas 20 veces menos. Por cada uno de los 14.758 alojamientos hace tres cosas: convierte con `json` un texto de unos 570 caracteres en una lista de Python (32 comodidades en promedio), une esa lista en un solo texto y busca en él ocho expresiones regulares. Los demás pasos son operaciones vectorizadas sobre columnas numéricas o categóricas.

Este es el paso donde una optimización tendría efecto real. En la celda siguiente comparamos dos implementaciones.

In [28]:
import json
import re

# Datos reales tal como llegan al extractor: se aplican todos los pasos anteriores a él
pasos = construir_pipeline().pasos_ejecutados()
indice = next(i for i, p in enumerate(pasos) if isinstance(p, ExtractorComodidades))
antes_del_extractor = datos.copy()
for paso in pasos[:indice]:
    antes_del_extractor = paso.ajustar_transformar(antes_del_extractor)
amenities = antes_del_extractor["amenities"]


def comodidades_con_json(columna):
    """Versión A (la actual): convierte cada texto en lista con json, la une y busca los patrones."""
    listas = columna.apply(json.loads)
    texto = listas.apply(lambda lista: "|".join(lista).lower())
    salida = pd.DataFrame({"n_comodidades": listas.str.len()})
    for nueva, patron in COMODIDADES.items():
        salida[nueva] = texto.str.contains(patron, regex=True).astype(int)
    return salida


def comodidades_sin_json(columna):
    """Versión B: busca los patrones directamente en el texto, sin crear listas de Python."""
    texto = columna.str.lower()
    # Cada separador '", "' divide dos elementos; la lista vacía "[]" tiene cero
    conteo = (texto.str.count(re.escape('", "')) + 1).where(texto != "[]", 0)
    salida = pd.DataFrame({"n_comodidades": conteo})
    for nueva, patron in COMODIDADES.items():
        salida[nueva] = texto.str.contains(patron, regex=True).astype(int)
    return salida


# Antes de comparar: las dos deben dar EXACTAMENTE lo mismo
pd.testing.assert_frame_equal(comodidades_con_json(amenities), comodidades_sin_json(amenities))
print(f"Las dos versiones dan el mismo resultado sobre los {len(amenities)} alojamientos.\n")

t_a = min(timeit.repeat(lambda: comodidades_con_json(amenities), number=3, repeat=5)) / 3
t_b = min(timeit.repeat(lambda: comodidades_sin_json(amenities), number=3, repeat=5)) / 3
_, _, mem_a = medir(comodidades_con_json, amenities)
_, _, mem_b = medir(comodidades_sin_json, amenities)

print(f"A · con json : {t_a*1000:.1f} ms   {mem_a:.1f} MB")
print(f"B · sin json : {t_b*1000:.1f} ms   {mem_b:.1f} MB")
print(f"\nB es {t_a / t_b:.2f} veces más rápida y usa {mem_b / mem_a:.2f} veces la memoria de A.")

Las dos versiones dan el mismo resultado sobre los 14758 alojamientos.

A · con json : 551.2 ms   42.6 MB
B · sin json : 584.6 ms   9.9 MB

B es 0.94 veces más rápida y usa 0.23 veces la memoria de A.


**Cómo se lee.** El `assert_frame_equal` confirma que las dos versiones producen el mismo conteo y los mismos ocho indicadores en los 14.758 alojamientos. La diferencia está en cómo llegan a ese resultado:

- **A (con json)** convierte el texto de cada alojamiento en una lista de Python, la une en un solo texto y busca allí los ocho patrones.
- **B (sin json)** busca los patrones directamente en el texto original y cuenta las comodidades por sus separadores.

**Las mediciones muestran un intercambio entre tiempo y memoria:**

- **Tiempo: prácticamente empatan** (A 517 ms, B 542 ms; B es cerca de un 5 % más lenta). B se ahorra la conversión con `json` y la unión de cada lista, pero recorre el texto original, que es más largo porque conserva comillas y separadores, y además necesita una búsqueda extra para contar los separadores. Las ocho búsquedas de patrones, que las dos versiones hacen, siguen siendo la parte más cara. Ambas son O(n × L), donde n es el número de alojamientos y L el largo del texto.
- **Memoria: B usa cerca de una cuarta parte** (9,9 MB contra 42,6 MB). A crea una lista por alojamiento con unas 32 comodidades: cerca de 475 mil objetos de Python en total. B no crea esas estructuras y trabaja sobre una sola copia en minúsculas del texto.

**Decisión.** Mantenemos A en `ExtractorComodidades`. Con 14.758 alojamientos, sus 43 MB no son una restricción para un computador actual, el tiempo es prácticamente el mismo, y A interpreta el JSON fila por fila, lo que permite detectar filas mal formadas: agregamos el manejo de ese error, que ahora lanza `ProyectoError` indicando la fila. B sería la opción preferible si la memoria pasara a ser la limitante, por ejemplo al procesar varias capturas de Inside Airbnb al mismo tiempo.

---
# 9. Patrón de diseño: Strategy

Aplicamos un solo patrón, **Strategy**, en la imputación de valores faltantes. En la Fase 2 comparamos tres formas de imputar (mediana, media y mediana por grupo), y cada una era un bloque de código distinto: cambiar de una a otra obligaba a reescribir. Con Strategy, el imputador (`ImputadorFlexible`, en `src/transformadores.py`) decide **cuándo** aprender y aplicar el valor de relleno, y la estrategia que recibe (`src/estrategias.py`) decide **cómo** calcularlo.

In [29]:
# El patrón Strategy vive en src/: las estrategias en src/estrategias.py
# y el imputador que las usa en src/transformadores.py
from src.estrategias import PorMediana, PorMedia, PorMedianaDeGrupo
from src.transformadores import ImputadorFlexible

# Se comparan las tres estrategias sobre beds sin modificar el imputador:
# solo cambia la estrategia que recibe
desv_original = datos["beds"].std()
comparacion = []
for estrategia in [PorMediana(), PorMedia(), PorMedianaDeGrupo("room_type")]:
    salida = ImputadorFlexible("beds", estrategia).ajustar_transformar(datos)
    comparacion.append({
        "estrategia": estrategia.etiqueta,
        "desv_antes": round(desv_original, 4),
        "desv_despues": round(salida["beds"].std(), 4),
        "cambio_pct": round((salida["beds"].std() - desv_original) / desv_original * 100, 2),
    })

pd.DataFrame(comparacion)

,estrategia,desv_antes,desv_despues,cambio_pct
0,mediana,1.842,1.8321,-0.54
1,media,1.842,1.8320,-0.54
2,mediana por grupo,1.842,1.8328,-0.49


In [30]:
# Evidencia: el pipeline del proyecto imputa con ImputadorFlexible y la estrategia por mediana
for paso in construir_pipeline().pasos_ejecutados():
    if isinstance(paso, ImputadorFlexible):
        print(f"{paso.nombre:<36} estrategia: {paso.estrategia.etiqueta}")

ImputadorFlexible(accommodates)      estrategia: mediana
ImputadorFlexible(latitude)          estrategia: mediana
ImputadorFlexible(bedrooms)          estrategia: mediana
ImputadorFlexible(beds)              estrategia: mediana
ImputadorFlexible(minimum_nights)    estrategia: mediana
ImputadorFlexible(maximum_nights)    estrategia: mediana


**Nota sobre estos números.** En el archivo completo, `beds` tiene 203 faltantes de 18.904 filas (cerca de un 1 %). Con tan pocos huecos, las tres estrategias cambian la desviación estándar en menos de un 1 % y son prácticamente equivalentes. Es la misma conclusión de la comparación de imputaciones de la Fase 2: cuando las estrategias no se distinguen, se declara y se elige la más simple, que es la mediana, coherente además con la asimetría de las variables del estudio.

**Decisión.** Las tres estrategias producen resultados prácticamente equivalentes, así que el pipeline del proyecto usa `PorMediana`: es la más simple y es coherente con la asimetría de las variables del estudio. La comparación se hizo sin modificar `ImputadorFlexible` ni el `Pipeline`: solo cambió la estrategia que se le entrega. Esa es la función del patrón.

## ¿Por qué Strategy en nuestro proyecto?

- **El problema que resuelve:** la imputación es el único paso del pipeline con varias alternativas reales, las tres que comparamos en la Fase 2. Sin el patrón, cambiar de estrategia obligaba a editar el imputador o a duplicarlo.
- **Qué gana el proyecto:** agregar una estrategia nueva, por ejemplo la mediana por comuna, es escribir una clase de pocas líneas en `src/estrategias.py`; ni `ImputadorFlexible` ni `Pipeline` cambian.
- **Por qué no usamos otros patrones:** Factory, Observer y Singleton no resolvían un problema concreto del proyecto. La configuración ya se entrega de forma explícita desde el cuaderno, y el registro de parámetros se genera en `parametros_f3.csv`. Agregarlos solo sumaba complejidad.

---
---
# 10. El pipeline construido desde la configuración

`construir_pipeline(config)`, en `src/pipeline.py`, arma los 24 pasos a partir de la celda de configuración: ningún módulo tiene nombres de columna escritos a mano. Para trabajar con otra captura o con otra ciudad basta con cambiar esa celda.

In [31]:
pipeline_config = construir_pipeline()
print(f"Pipeline construido con {len(pipeline_config)} pasos desde la configuración:\n")
print(pipeline_config.resumen().to_string(index=False))

Pipeline construido con 24 pasos desde la configuración:

 orden                                       paso                clase
     1                     ConversorPrecio(price)      ConversorPrecio
     2   FiltroFilas(solo alojamientos completos)          FiltroFilas
     3             SelectorColumnas(11 variables)     SelectorColumnas
     4                   Acotador(maximum_nights)             Acotador
     5       FiltroFilas(objetivos sin faltantes)          FiltroFilas
     6                    MarcadorNulos(bedrooms)        MarcadorNulos
     7            ImputadorFlexible(accommodates)    ImputadorFlexible
     8                ImputadorFlexible(latitude)    ImputadorFlexible
     9                ImputadorFlexible(bedrooms)    ImputadorFlexible
    10                    ImputadorFlexible(beds)    ImputadorFlexible
    11          ImputadorFlexible(minimum_nights)    ImputadorFlexible
    12          ImputadorFlexible(maximum_nights)    ImputadorFlexible
    13          Fil

In [32]:
# Se ajusta con entrenamiento y se aplica a prueba, sin fuga de datos
pipeline_config.ajustar(entrenamiento)
entrenamiento_listo = pipeline_config.transformar(entrenamiento)
prueba_lista = pipeline_config.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, "->", entrenamiento_listo.shape)
print("Prueba       :", prueba.shape, "->", prueba_lista.shape)

# Las dos salidas deben tener exactamente las mismas columnas y en el mismo orden
assert list(entrenamiento_listo.columns) == list(prueba_lista.columns), \
    "Entrenamiento y prueba quedaron con columnas distintas"
print("\nAmbos conjuntos tienen las mismas columnas, en el mismo orden.")

Entrenamiento: (15123, 90) -> (11788, 50)
Prueba       : (3781, 90) -> (2970, 50)

Ambos conjuntos tienen las mismas columnas, en el mismo orden.


**Esa última comprobación es más importante de lo que parece.** Si el codificador
aprendiera el vocabulario de cada conjunto por separado, entrenamiento y prueba
quedarían con columnas distintas y cualquier modelo posterior fallaría. Como el
vocabulario se aprende una sola vez en el ajuste, eso no puede pasar.

---
# 11. El resultado: cómo queda el código y cómo quedan los datos

Esta sección cierra el recorrido. Muestra dos cosas: **cómo se ve el proyecto después de
aplicar POO** y **qué conjunto queda listo para la Fase 4**.

## 11.1 El código, antes y después

**Antes (Fase 2).** El pipeline era una secuencia de celdas con funciones. Funcionaba, pero el orden estaba implícito y cada paso calculaba sus parámetros sobre el conjunto completo:

```python
df_crudo["price"] = pd.to_numeric(df_crudo["price"].str.replace("$", "")...)
df = df_crudo[df_crudo["room_type"] == "Entire home/apt"][variables_estudio]
df["maximum_nights"] = df["maximum_nights"].clip(upper=1125)
df = df.dropna(subset=OBJETIVOS)
for col in columnas_imputar:
    df = imputar_nulos_numericos(df, col, estrategia="mediana")
...
df = codificar_one_hot(df, "neighbourhood_cleansed", nombres_comunas)
df, scaler = escalar_caracteristicas(df, columnas_a_escalar)
```

**Después (Fase 3).** El mismo trabajo, en cuatro líneas:

```python
pipeline = construir_pipeline()
pipeline.ajustar(entrenamiento)
entrenamiento_listo = pipeline.transformar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)
```

Lo que cambió no es el resultado (la sección 6 lo comprueba): ahora cada paso es una pieza con nombre, que se puede probar, reemplazar y reutilizar sin tocar las demás, y que aprende sus parámetros solo del conjunto de entrenamiento

## 11.2 La arquitectura, generada desde el propio código

En vez de escribir a mano la tabla de arquitectura para el informe, se puede producir
leyendo las clases que existen. Así no queda desactualizada.

In [33]:
def documentar_arquitectura():
    """Genera la tabla de arquitectura a partir de las clases definidas.

    __subclasses__() devuelve las clases que heredan de Transformador. Es la
    misma información que Python usa para resolver la herencia, así que la
    tabla refleja el código real y no lo que creemos que hay.
    """
    filas = [{
        "componente": "Transformador",
        "rol": "clase base",
        "responsabilidad": "Define el contrato y controla el estado de cada paso",
        "archivo": "src/transformadores.py",
    }]

    for clase in Transformador.__subclasses__():
        resumen = (clase.__doc__ or "sin documentar").strip().split("\n")[0]
        filas.append({
            "componente": clase.__name__,
            "rol": "clase hija",
            "responsabilidad": resumen,
            "archivo": (f"src/{clase.__module__.split('.')[-1]}.py"
                        if clase.__module__.startswith("src.") else "cuaderno (ejemplo)"),
        })

    filas += [
        {"componente": "EstrategiaImputacion y derivadas", "rol": "estrategias (Strategy)",
         "responsabilidad": "Calculan el valor de relleno: mediana, media o mediana por grupo",
         "archivo": "src/estrategias.py"},
        {"componente": "Pipeline", "rol": "orquestador",
         "responsabilidad": "Encadena los pasos y controla el orden de ajuste",
         "archivo": "src/pipeline.py"},
        {"componente": "construir_pipeline", "rol": "constructor del pipeline",
         "responsabilidad": "Arma el pipeline desde la celda de configuración",
         "archivo": "src/pipeline.py"},
        {"componente": "cargar", "rol": "carga",
         "responsabilidad": "Lee el archivo crudo y verifica su esquema",
         "archivo": "src/carga.py"},
        {"componente": "medir", "rol": "utilidad",
         "responsabilidad": "Mide tiempo y memoria de cualquier función",
         "archivo": "src/medicion.py"},
    ]
    return pd.DataFrame(filas)


arquitectura = documentar_arquitectura()
print(arquitectura.to_string(index=False))

                      componente                      rol                                                             responsabilidad                archivo
                   Transformador               clase base                        Define el contrato y controla el estado de cada paso src/transformadores.py
                ImputadorMediana               clase hija                    Rellena los nulos con la mediana aprendida en el ajuste. src/transformadores.py
               ImputadorFlexible               clase hija Rellena los nulos con el valor que calcula su estrategia (patrón Strategy). src/transformadores.py
              CodificadorNominal               clase hija              Convierte una columna de categorías en columnas 0/1 (One-Hot). src/transformadores.py
                EscaladorRobusto               clase hija  Centra en la mediana y divide por el rango intercuartílico (RobustScaler). src/transformadores.py
              EliminadorColumnas               clase hija 

## 11.3 El conjunto procesado: qué queda listo

Acá se ejecuta el pipeline completo y se revisa el resultado con el detalle que la
Fase 4 va a necesitar.

In [34]:
# Pipeline definitivo, construido desde la configuración
pipeline_final = construir_pipeline()
pipeline_final.ajustar(entrenamiento)

entrenamiento_final = pipeline_final.transformar(entrenamiento)
prueba_final = pipeline_final.transformar(prueba)

print("CONJUNTO PROCESADO")
print("=" * 58)
print(f"Entrenamiento : {entrenamiento_final.shape[0]:>6} filas x {entrenamiento_final.shape[1]:>3} columnas")
print(f"Prueba        : {prueba_final.shape[0]:>6} filas x {prueba_final.shape[1]:>3} columnas")
print(f"Partida desde : {datos.shape[0]:>6} filas x {datos.shape[1]:>3} columnas")
print(f"\nColumnas finales: {entrenamiento_final.shape[1]} (las mismas 50 de la Fase 2)")
entrenamiento_final.head(3)

CONJUNTO PROCESADO
Entrenamiento :  11788 filas x  50 columnas
Prueba        :   2970 filas x  50 columnas
Partida desde :  18904 filas x  90 columnas

Columnas finales: 50 (las mismas 50 de la Fase 2)


,price,estimated_occupancy_l365d,accommodates,bedrooms,beds,latitude,minimum_nights,maximum_nights,bedrooms_imputado,n_comodidades,...,comuna_Pudahuel,comuna_Quilicura,comuna_Quinta_Normal,comuna_Recoleta,comuna_Renca,comuna_San_Joaquín,comuna_San_Miguel,comuna_Santiago,comuna_Vitacura,comuna_Ñuñoa
2388,84947.0,36,0.5,1.0,0.5,-0.512463,0.0,1.0,0,0.454545,...,0,0,0,0,0,0,0,0,0,1
9680,90000.0,0,-0.5,0.0,-0.5,1.139384,3.0,0.0,0,-1.227273,...,0,0,0,0,0,0,0,0,1,0
4489,43500.0,0,-0.5,0.0,-0.5,-0.451679,-1.0,1.0,0,0.727273,...,0,0,0,0,0,0,0,1,0,0


Se describe finalmente una función para caracterizar el esquema:

In [35]:
# Esquema final: qué tipo tiene cada columna y de dónde salió
def describir_esquema(df, originales):
    """Clasifica cada columna del resultado según su origen."""
    filas = []
    for columna in df.columns:
        if columna in originales:
            origen = "original"
        elif "_" in columna and columna.split("_")[0] in originales:
            origen = f"derivada de {columna.split('_')[0]}"
        else:
            origen = "derivada"
        filas.append({
            "columna": columna,
            "tipo": str(df[columna].dtype),
            "origen": origen,
            "nulos": int(df[columna].isna().sum()),
        })
    return pd.DataFrame(filas)


esquema = describir_esquema(entrenamiento_final, set(datos.columns))
print(f"Total de columnas: {len(esquema)}")
print(f"Originales conservadas: {(esquema['origen'] == 'original').sum()}")
print(f"Derivadas de la codificación: {(esquema['origen'] != 'original').sum()}")
print(f"Columnas con nulos: {(esquema['nulos'] > 0).sum()}")
print()
esquema.head(12)

Total de columnas: 50
Originales conservadas: 8
Derivadas de la codificación: 42
Columnas con nulos: 0



,columna,tipo,origen,nulos
0,price,float64,original,0
1,estimated_occupancy_l365d,int64,original,0
2,accommodates,float64,original,0
3,bedrooms,float64,original,0
4,beds,float64,original,0
5,latitude,float64,original,0
6,minimum_nights,float64,original,0
7,maximum_nights,float64,original,0
8,bedrooms_imputado,int64,derivada de bedrooms,0
9,n_comodidades,float64,derivada,0


## 11.4 Del cuaderno a los módulos

El criterio de diseño estructurado no se cumple con clases escritas dentro del cuaderno: el código reutilizable vive en `src/` y el cuaderno lo **importa** en cada sección donde lo usa.

| Archivo | Qué contiene | Dónde se importa |
|---|---|---|
| `src/carga.py` | `leer_archivo`, `verificar_esquema` y `cargar` | Carga del conjunto |
| `src/transformadores.py` | `Transformador` y las 10 clases de pasos que heredan de ella | Sección 3 |
| `src/pipeline.py` | `Pipeline` y `construir_pipeline(config)` | Sección 4 |
| `src/utilidades.py` | Definir la `Excepción` del proyecto | Sección 6 |
| `src/medicion.py` | `medir`: tiempo y memoria pico | Sección 8 |
| `src/estrategias.py` | Estrategias de imputación (patrón `Strategy`) | Sección 9 |.

Los módulos no guardan rutas ni nombres de columna: el cuaderno se los entrega (`RUTA_DATOS`, `COLUMNAS_ESPERADAS`, `CONFIG`), así la configuración vive en un solo lugar. Las clases que siguen escritas en el cuaderno son ejemplos didácticos.

La celda siguiente comprueba de dónde viene cada componente que usa el pipeline. La equivalencia con la Fase 2 ya quedó demostrada en la sección 6, que usa exactamente estos módulos.

In [36]:
import inspect
from pathlib import Path

componentes = [cargar, Transformador, ImputadorMediana, CodificadorNominal,
               EscaladorRobusto, EliminadorColumnas, ConversorPrecio, FiltroFilas,
               SelectorColumnas, Acotador, MarcadorNulos, ExtractorComodidades,
               ImputadorFlexible, PorMediana, PorMedia, PorMedianaDeGrupo, Pipeline, construir_pipeline_desde_config, medir]

origen = pd.DataFrame([{
    "componente": c.__name__,
    "módulo": c.__module__,
    "archivo": "src/" + Path(inspect.getfile(c)).name,
} for c in componentes])

# Si algún componente se hubiera redefinido en el cuaderno, su módulo sería __main__
assert origen["módulo"].str.startswith("src.").all(), "Hay componentes definidos en el cuaderno"

print(origen.to_string(index=False))
print(f"\nLos {len(origen)} componentes del pipeline se importan desde src/.")

          componente              módulo                archivo
              cargar           src.carga           src/carga.py
       Transformador src.transformadores src/transformadores.py
    ImputadorMediana src.transformadores src/transformadores.py
  CodificadorNominal src.transformadores src/transformadores.py
    EscaladorRobusto src.transformadores src/transformadores.py
  EliminadorColumnas src.transformadores src/transformadores.py
     ConversorPrecio src.transformadores src/transformadores.py
         FiltroFilas src.transformadores src/transformadores.py
    SelectorColumnas src.transformadores src/transformadores.py
            Acotador src.transformadores src/transformadores.py
       MarcadorNulos src.transformadores src/transformadores.py
ExtractorComodidades src.transformadores src/transformadores.py
   ImputadorFlexible src.transformadores src/transformadores.py
          PorMediana     src.estrategias     src/estrategias.py
            PorMedia     src.estrategias

## 11.5 Verificación de entrega

Antes de pasar a la Fase 4 conviene comprobar cinco condiciones. Si alguna falla, el
problema es de esta fase y corregirlo acá cuesta mucho menos que descubrirlo después.

In [37]:
def verificar_entrega(entrenamiento_final, prueba_final, total_final, objetivos):
    """Cinco comprobaciones antes de entregar el conjunto a la Fase 4."""
    controles = []

    # 1. La partición no perdió ni duplicó filas: entrenamiento + prueba = total procesado
    obtenidas = len(entrenamiento_final) + len(prueba_final)
    controles.append(("Partición sin filas perdidas ni duplicadas",
                      obtenidas == len(total_final), f"{obtenidas} de {len(total_final)}"))

    # 2. Entrenamiento y prueba tienen el mismo esquema
    mismo_esquema = list(entrenamiento_final.columns) == list(prueba_final.columns)
    controles.append(("Mismo esquema en ambos conjuntos",
                      mismo_esquema, f"{entrenamiento_final.shape[1]} columnas"))

    # 3. No quedan nulos en ninguno de los dos conjuntos
    nulos = int(entrenamiento_final.isna().sum().sum() + prueba_final.isna().sum().sum())
    controles.append(("Sin valores nulos", nulos == 0, f"{nulos} nulos"))

    # 4. Los objetivos sobrevivieron completos
    objetivos_ok = all(o in entrenamiento_final.columns
                       and entrenamiento_final[o].isna().sum() == 0 for o in objetivos)
    controles.append(("Objetivos presentes y completos", objetivos_ok, ", ".join(objetivos)))

    # 5. Todo quedó numérico y sin identificador
    numerico = entrenamiento_final.select_dtypes(exclude="number").empty
    sin_id = COLUMNA_ID not in entrenamiento_final.columns
    controles.append(("Todo numérico y sin identificador", numerico and sin_id, COLUMNA_ID))

    tabla = pd.DataFrame(
        [{"control": c[0], "estado": "OK" if c[1] else "FALLA", "detalle": c[2]}
         for c in controles]
    )
    return tabla, all(c[1] for c in controles)


total_final = pipeline_final.transformar(datos)
controles, aprobado = verificar_entrega(entrenamiento_final, prueba_final,
                                        total_final, OBJETIVOS)
print(controles.to_string(index=False))
print("\n" + ("ENTREGA APROBADA: el conjunto puede pasar a la Fase 4."
               if aprobado else
               "ENTREGA RECHAZADA: revisar los controles en estado FALLA."))

                                   control estado                          detalle
Partición sin filas perdidas ni duplicadas     OK                   14758 de 14758
          Mismo esquema en ambos conjuntos     OK                      50 columnas
                         Sin valores nulos     OK                          0 nulos
           Objetivos presentes y completos     OK price, estimated_occupancy_l365d
         Todo numérico y sin identificador     OK                               id

ENTREGA APROBADA: el conjunto puede pasar a la Fase 4.


## 11.6 Lo que se entrega a la Fase 4

La Fase 4 no debe volver a limpiar ni transformar. Recibe cuatro artefactos y trabaja
sobre ellos.

In [38]:
import json

def entregar_a_fase4(entrenamiento_final, prueba_final, pipeline, esquema,
                     carpeta=DIR_PROCESADO):
    """Escribe los cuatro artefactos que recibe la Fase 4."""
    os.makedirs(carpeta, exist_ok=True)
    entregables = {}

    # 1. Los conjuntos listos para analizar
    ruta_ent = os.path.join(carpeta, "entrenamiento_f3.csv")
    ruta_pru = os.path.join(carpeta, "prueba_f3.csv")
    entrenamiento_final.to_csv(ruta_ent, index=False)
    prueba_final.to_csv(ruta_pru, index=False)
    entregables["entrenamiento"] = ruta_ent
    entregables["prueba"] = ruta_pru

    # 2. El diccionario del conjunto resultante
    ruta_dic = os.path.join(carpeta, "diccionario_f3.csv")
    esquema.to_csv(ruta_dic, index=False)
    entregables["diccionario"] = ruta_dic

    # 3. Los parámetros aprendidos: es lo que permite reproducir el resultado
    ruta_par = os.path.join(carpeta, "parametros_f3.csv")
    pd.DataFrame([
        {"orden": i, "paso": p.nombre,
         "parametros": json.dumps(p.parametros, ensure_ascii=False)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ]).to_csv(ruta_par, index=False)
    entregables["parametros"] = ruta_par

    return entregables


entregables = entregar_a_fase4(entrenamiento_final, prueba_final,
                               pipeline_final, esquema)

print("ARTEFACTOS ENTREGADOS A LA FASE 4")
print("=" * 58)
for nombre, ruta in entregables.items():
    tamano = os.path.getsize(ruta) / 1024
    print(f"  {nombre:<16} {ruta:<40} {tamano:>7.1f} KB")

ARTEFACTOS ENTREGADOS A LA FASE 4
  entrenamiento    c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\entrenamiento_f3.csv  1806.6 KB
  prueba           c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\prueba_f3.csv   456.1 KB
  diccionario      c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\diccionario_f3.csv     1.7 KB
  parametros       c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\parametros_f3.csv     2.7 KB


### Parámetros en formato JSON: cómo se revierte el escalado en la Fase 4

`EscaladorRobusto` transforma siete predictores (`accommodates`, `latitude`, `bedrooms`, `beds`, `minimum_nights`, `maximum_nights` y `n_comodidades`) con la fórmula `z = (x − mediana) / RIC`. Para interpretarlos y graficarlos en sus unidades reales, la Fase 4 debe aplicar la operación inversa:

**x = z × RIC + mediana**

La mediana y el RIC de cada variable, aprendidos con el conjunto de entrenamiento, quedan en `parametros_f3.csv`. En una primera versión los guardábamos con `str()`, que produce un diccionario de Python con comillas simples (`{'mediana': 3.0, 'ric': 2.0}`). Ese texto no se puede leer con `json.loads` y obligaba a la Fase 4 a interpretarlo con métodos menos seguros. Ahora se guardan con `json.dumps` (`{"mediana": 3.0, "ric": 2.0}`), un formato estándar que la Fase 4 lee directamente.

Al abrir el archivo, pandas muestra la columna `parametros` como texto, porque en un CSV todo valor es texto. Lo importante es que ese texto es JSON válido: `json.loads` lo convierte en un diccionario con valores numéricos. La celda siguiente comprueba que la inversión recupera exactamente los valores originales.

In [39]:
# Comprobación: la Fase 4 puede revertir el escalado con los parámetros guardados
import json
import numpy as np

# 1. Leer los parámetros de los escaladores tal como los leerá la Fase 4
parametros = pd.read_csv(entregables["parametros"])
escaladores = {
    fila["paso"]: json.loads(fila["parametros"])
    for _, fila in parametros.iterrows()
    if fila["paso"].startswith("EscaladorRobusto")
}

# 2. Referencia: el mismo pipeline ya ajustado, aplicado sin los pasos de escalado
sin_escalar = entrenamiento.copy()
for paso in pipeline_final.pasos_ejecutados():
    if not isinstance(paso, EscaladorRobusto):
        sin_escalar = paso.transformar(sin_escalar)

# 3. Invertir cada variable escalada del archivo guardado y compararla con la referencia
entrenamiento_leido = pd.read_csv(entregables["entrenamiento"])
for paso, p in escaladores.items():
    columna = paso[len("EscaladorRobusto("):-1]
    original = entrenamiento_leido[columna] * p["ric"] + p["mediana"]
    assert np.allclose(original.values, sin_escalar[columna].values), columna
    print(f"{columna:<16} mediana = {p['mediana']:<10g} RIC = {p['ric']:<8.4g} -> inversión exacta")

print("\nLos 7 predictores escalados se recuperan en sus unidades originales.")

accommodates     mediana = 3          RIC = 2        -> inversión exacta
latitude         mediana = -33.4396   RIC = 0.02978  -> inversión exacta
bedrooms         mediana = 1          RIC = 1        -> inversión exacta
beds             mediana = 2          RIC = 2        -> inversión exacta
minimum_nights   mediana = 2          RIC = 1        -> inversión exacta
maximum_nights   mediana = 365        RIC = 760      -> inversión exacta
n_comodidades    mediana = 34         RIC = 22       -> inversión exacta

Los 7 predictores escalados se recuperan en sus unidades originales.


**Cómo se lee.** Para cada variable se muestran la mediana y el RIC aprendidos con el entrenamiento. El `assert` confirma que, al aplicar `x = z × RIC + mediana` sobre el archivo guardado, se obtienen exactamente los mismos valores que antes de escalar. La Fase 4 puede entonces trabajar con los datos escalados y volver a unidades reales (huéspedes, noches, número de comodidades) cuando necesite interpretar o graficar.

### El contrato entre fases

| Artefacto | Qué contiene | Para qué lo usa la Fase 4 |
|---|---|---|
| `entrenamiento_f3.csv` | El conjunto procesado de entrenamiento | Ajustar y explorar |
| `prueba_f3.csv` | El conjunto procesado de prueba | Evaluar, con el mismo esquema |
| `diccionario_f3.csv` | Columna, tipo, origen y nulos | Saber qué significa cada variable |
| `parametros_f3.csv` | Lo que aprendió cada paso | Reproducir el resultado o auditarlo |

**Lo que la Fase 4 NO debe hacer:** volver a imputar, volver a codificar ni volver a
escalar. Si lo hiciera sobre el conjunto ya transformado, escalaría dos veces y los
resultados dejarían de tener sentido.

**Lo que sí debe hacer:** leer estos archivos, explorar, visualizar y comunicar. Y si
necesita una transformación nueva, agregarla como un paso más del pipeline de la Fase 3,
no como código suelto en su propio cuaderno. Esa es la ventaja de haber modularizado.

Se deja una celda de código que realiza una comprobación final respecto a lo que queda listo para la fase 4:

In [40]:
# Comprobación final: lo que se guardó se puede volver a leer sin sorpresas
releido = pd.read_csv(entregables["entrenamiento"])

print("Verificación de ida y vuelta")
print(f"  Antes de guardar : {entrenamiento_final.shape}")
print(f"  Después de leer  : {releido.shape}")
print(f"  Mismas columnas  : {list(releido.columns) == list(entrenamiento_final.columns)}")
print(f"  Sin nulos nuevos : {int(releido.isna().sum().sum()) == int(entrenamiento_final.isna().sum().sum())}")
print("\nEl conjunto está listo para la Fase 4.")

Verificación de ida y vuelta
  Antes de guardar : (11788, 50)
  Después de leer  : (11788, 50)
  Mismas columnas  : True
  Sin nulos nuevos : True

El conjunto está listo para la Fase 4.


---
# 12. Ejercicios: dónde quedó resuelto cada uno

| Ejercicio | Criterio que alimenta | Dónde está resuelto |
|---|---|---|
| 1 · Encapsular un paso del pipeline | Programación orientada a objetos | Sección 3: `ConversorPrecio`, `FiltroFilas`, `SelectorColumnas`, `Acotador`, `MarcadorNulos` y `ExtractorComodidades` |
| 2 · Armar el pipeline completo | Diseño estructurado y preprocesamiento | Sección 4: `construir_pipeline()`, los 24 pasos de la Fase 2 |
| 3 · Comparar dos implementaciones | Eficiencia y optimización | Sección 6: escalado con bucle vs vectorizado, con `timeit` y `tracemalloc` |
| 4 · Validar en tres escenarios | Validación técnica | Sección 6: caso normal, casos límite y excepciones |
| 5 · Patrón y equivalencia con la F2 | Documentación de arquitectura y validación técnica | Patrón declarado: sección 9 (Strategy). Equivalencia: sección 8, `assert_frame_equal` contra `airbnb_procesado.csv` |

---
# Cierre

## Resumen de lo implementado

| Concepto | Dónde se ve en este cuaderno |
|---|---|
| Encapsulamiento | `_parametros`, `_ajustado` y propiedades de solo lectura (sección 2) |
| Herencia | Las clases de pasos (`ImputadorMediana`, `ImputadorFlexible`, `CodificadorNominal`, `EscaladorRobusto`, `FiltroFilas`, etc.) heredan de `Transformador` (sección 3) |
| Polimorfismo | El `Pipeline` recorre los 24 pasos sin preguntar su tipo (secciones 3 y 4) |
| Recursividad | `aplanar()` y el contraste entre Fibonacci ingenua y memoizada (sección 7) |
| Eficiencia | Escalado con bucle y vectorizado, tabla de crecimiento, medición de los 24 pasos y comparación con y sin `json` del extractor (sección 8) |
| Equivalencia con F2 | `assert_frame_equal` contra `airbnb_procesado.csv` (sección 6) |
| Modularidad | Todo el pipeline se importa desde `src/` (sección 11.4) |
| Patrón de diseño | Strategy: `ImputadorFlexible` y `src/estrategias.py` (sección 9) |
| Entrega a la Fase 4 | Cuatro artefactos, parámetros en JSON y verificación de que el escalado se puede revertir (sección 11.6) |

## Bibliografía (APA 7)

Grus, J. (2019). Data science from scratch (2.ª ed.). O’Reilly Media. 

McKinney, W. (2022). Python for data analysis (3.ª ed.). O’Reilly Media. 

Python Software Foundation. (s. f.). time — Time access and conversions. https://docs.python.org/3/library/time.html 

Python Software Foundation. (s. f.). tracemalloc — Trace memory allocations. https://docs.python.org/3/library/tracemalloc.html 

Rule, A., Birmingham, A., Zuniga, C., Altintas, I., Huang, S.-C., Knight, R., Moshiri, N., Nguyen, M. H., Rosenthal, S. B., Pérez, F., & Rose, P. W. (2019). Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks. PLOS Computational Biology, 15(7), e1007007. 

The pandas development team. (s. f.). pandas documentation. https://pandas.pydata.org/docs/ 

 